# NvDiffRast Texture Refinement

**Input**: OpenMVS textured mesh (5K-10K faces) + keyframe images + ARKit poses (COLMAP format)  
**Output**: Refined texture atlas with reduced seam artifacts  
**Runtime**: ~2-5 min on T4

Differentiable rendering optimizes the texture so it looks correct from all camera viewpoints simultaneously,
replacing the seam leveling step that crashes on OpenMVS v2.3.0.

In [ ]:
# Cell 1: Install nvdiffrast (torch + CUDA already on Colab)
!pip install -q nvdiffrast ninja imageio

import torch
print(f"PyTorch {torch.__version__}, CUDA {torch.version.cuda}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

import nvdiffrast.torch as dr
print("nvdiffrast OK")

In [ ]:
# Cell 2: Upload data
# Expected zip contents (from prepare_nvdiffrast_input.sh):
#   images/          - keyframe JPEGs
#   sparse/          - cameras.txt, images.txt
#   textured.obj     - OpenMVS textured mesh
#   textured.mtl
#   textured_*_map_Kd.jpg  - texture atlas

import zipfile, os, glob
from google.colab import files

print("Upload nvdiffrast_input.zip")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

WORK = '/content/work'
os.makedirs(WORK, exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall(WORK)

# Find the OBJ file
obj_files = glob.glob(os.path.join(WORK, '*.obj'))
if not obj_files:
    obj_files = glob.glob(os.path.join(WORK, '**/*.obj'), recursive=True)
print(f"Found OBJ: {obj_files}")
print(f"Images: {len(glob.glob(os.path.join(WORK, 'images', '*.jpg')))}")

In [ ]:
# Cell 3: Helpers
import numpy as np
import torch
import torch.nn.functional as F
import nvdiffrast.torch as dr
from PIL import Image
from collections import OrderedDict


def load_obj(path):
    """Parse OBJ → unified vertex/UV buffers for nvdiffrast."""
    positions, texcoords = [], []
    face_pos, face_uv = [], []
    with open(path) as f:
        for line in f:
            s = line.strip().split()
            if not s:
                continue
            if s[0] == 'v' and len(s) >= 4:
                positions.append([float(s[1]), float(s[2]), float(s[3])])
            elif s[0] == 'vt' and len(s) >= 3:
                texcoords.append([float(s[1]), float(s[2])])
            elif s[0] == 'f':
                fp, ft = [], []
                for vert in s[1:]:
                    idx = vert.split('/')
                    fp.append(int(idx[0]) - 1)
                    ft.append(int(idx[1]) - 1 if len(idx) > 1 and idx[1] else 0)
                # Triangulate face fans (handles quads)
                for i in range(1, len(fp) - 1):
                    face_pos.append([fp[0], fp[i], fp[i + 1]])
                    face_uv.append([ft[0], ft[i], ft[i + 1]])

    positions = np.array(positions, dtype=np.float32)
    texcoords = np.array(texcoords, dtype=np.float32)
    face_pos = np.array(face_pos, dtype=np.int32)
    face_uv = np.array(face_uv, dtype=np.int32)

    # Unify: one vertex per unique (pos_idx, uv_idx) pair
    vert_map = {}
    uni_pos, uni_uv, uni_faces = [], [], []
    for fi in range(len(face_pos)):
        tri = []
        for vi in range(3):
            key = (face_pos[fi, vi], face_uv[fi, vi])
            if key not in vert_map:
                vert_map[key] = len(uni_pos)
                uni_pos.append(positions[key[0]])
                uni_uv.append(texcoords[key[1]])
            tri.append(vert_map[key])
        uni_faces.append(tri)

    uni_uv = np.array(uni_uv, dtype=np.float32)
    # OBJ UV origin = bottom-left; nvdiffrast texture origin = top-left → flip V
    uni_uv[:, 1] = 1.0 - uni_uv[:, 1]

    return (
        np.array(uni_pos, dtype=np.float32),
        uni_uv,
        np.array(uni_faces, dtype=np.int32),
    )


def quat_to_rot(qw, qx, qy, qz):
    """Quaternion (w,x,y,z) → 3x3 rotation."""
    return np.array([
        [1 - 2*(qy*qy + qz*qz), 2*(qx*qy - qz*qw), 2*(qx*qz + qy*qw)],
        [2*(qx*qy + qz*qw), 1 - 2*(qx*qx + qz*qz), 2*(qy*qz - qx*qw)],
        [2*(qx*qz - qy*qw), 2*(qy*qz + qx*qw), 1 - 2*(qx*qx + qy*qy)],
    ], dtype=np.float32)


def parse_colmap_cameras(path):
    """Parse cameras.txt → (fx, fy, cx, cy, W, H)."""
    with open(path) as f:
        for line in f:
            if line.startswith('#'):
                continue
            parts = line.strip().split()
            if len(parts) >= 8 and parts[1] == 'PINHOLE':
                W, H = int(parts[2]), int(parts[3])
                fx, fy, cx, cy = float(parts[4]), float(parts[5]), float(parts[6]), float(parts[7])
                return fx, fy, cx, cy, W, H
    raise ValueError("No PINHOLE camera found")


def parse_colmap_images(path):
    """Parse images.txt → {name: 4x4 cam_from_world (COLMAP convention)}."""
    poses = OrderedDict()
    with open(path) as f:
        lines = [l.strip() for l in f if l.strip() and not l.startswith('#')]
    for i in range(0, len(lines), 2):
        parts = lines[i].split()
        qw, qx, qy, qz = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        tx, ty, tz = float(parts[5]), float(parts[6]), float(parts[7])
        name = parts[9]
        R = quat_to_rot(qw, qx, qy, qz)
        T = np.eye(4, dtype=np.float32)
        T[:3, :3] = R
        T[:3, 3] = [tx, ty, tz]
        poses[name] = T
    return poses


def colmap_to_opengl_view(T_colmap):
    """COLMAP cam-from-world → OpenGL cam-from-world.
    COLMAP: X-right, Y-down, Z-forward
    OpenGL: X-right, Y-up, Z-backward"""
    flip = np.diag([1., -1., -1., 1.]).astype(np.float32)
    return flip @ T_colmap


def make_projection(fx, fy, cx, cy, W, H, near=0.05, far=50.0):
    """Pinhole intrinsics → OpenGL clip-space projection."""
    P = torch.zeros(4, 4)
    P[0, 0] = 2.0 * fx / W
    P[1, 1] = 2.0 * fy / H
    P[0, 2] = (W - 2.0 * cx) / W
    P[1, 2] = (2.0 * cy - H) / H
    P[2, 2] = -(far + near) / (far - near)
    P[2, 3] = -2.0 * far * near / (far - near)
    P[3, 2] = -1.0
    return P


def render(glctx, pos, faces, uv, tex, mvp, resolution):
    """Render textured mesh → (1,H,W,3) RGB, (1,H,W,1) mask."""
    V = pos.shape[0]
    pos_homo = torch.cat([pos, torch.ones(V, 1, device=pos.device)], dim=1)
    pos_clip = (mvp @ pos_homo.T).T.unsqueeze(0).contiguous()  # (1, V, 4)

    rast, rast_db = dr.rasterize(glctx, pos_clip, faces, resolution)
    texc, texc_db = dr.interpolate(
        uv.unsqueeze(0), rast, faces, rast_db=rast_db, diff_attrs='all'
    )
    color = dr.texture(tex, texc, texc_db,
                       filter_mode='linear-mipmap-linear',
                       boundary_mode='clamp')
    mask = (rast[..., 3:4] > 0).float()

    # Flip Y: OpenGL (origin bottom-left) → image (origin top-left)
    return color.flip(1), mask.flip(1)


def tv_loss(tex):
    """Total variation regularizer."""
    dx = (tex[:, :, 1:, :] - tex[:, :, :-1, :]).abs()
    dy = (tex[:, 1:, :, :] - tex[:, :-1, :, :]).abs()
    return (dx.mean() + dy.mean()) * 0.5


print("Helpers loaded.")

In [ ]:
# Cell 4: Load mesh, texture, poses, images
WORK = '/content/work'
RENDER_SCALE = 0.5  # Half-res for speed; full texture resolution is preserved

# --- Mesh ---
obj_path = sorted(glob.glob(os.path.join(WORK, '*.obj')))[0]
print(f"Mesh: {os.path.basename(obj_path)}")
positions, uvs, faces = load_obj(obj_path)
print(f"  {len(positions)} unified vertices, {len(faces)} faces")

# --- Texture ---
mtl_path = obj_path.replace('.obj', '.mtl')
tex_file = None
with open(mtl_path) as f:
    for line in f:
        if 'map_Kd' in line:
            tex_file = os.path.join(WORK, line.strip().split()[-1])
tex_img = np.array(Image.open(tex_file).convert('RGB'), dtype=np.float32) / 255.0
print(f"Texture: {tex_file} → {tex_img.shape}")

# --- Intrinsics ---
fx, fy, cx, cy, FULL_W, FULL_H = parse_colmap_cameras(
    os.path.join(WORK, 'sparse', 'cameras.txt'))
RW = int(FULL_W * RENDER_SCALE)
RH = int(FULL_H * RENDER_SCALE)
sfx, sfy = fx * RENDER_SCALE, fy * RENDER_SCALE
scx, scy = cx * RENDER_SCALE, cy * RENDER_SCALE
print(f"Intrinsics: fx={fx:.1f} fy={fy:.1f} | Render {RW}x{RH}")

# --- Poses ---
colmap_poses = parse_colmap_images(
    os.path.join(WORK, 'sparse', 'images.txt'))
print(f"Poses: {len(colmap_poses)}")

# --- Images (downsampled to render resolution) ---
ref_imgs = {}
view_mats = {}
for name, T_col in colmap_poses.items():
    img_path = os.path.join(WORK, 'images', name)
    if not os.path.exists(img_path):
        continue
    img = Image.open(img_path).convert('RGB').resize((RW, RH), Image.LANCZOS)
    ref_imgs[name] = np.array(img, dtype=np.float32) / 255.0
    view_mats[name] = colmap_to_opengl_view(T_col)

view_names = sorted(view_mats.keys())
print(f"Loaded {len(view_names)} keyframe images")

In [ ]:
# Cell 5: Setup GPU tensors + projection
device = torch.device('cuda')

t_pos = torch.tensor(positions, device=device)
t_uv = torch.tensor(uvs, device=device)
t_faces = torch.tensor(faces, device=device, dtype=torch.int32)

# Optimizable texture
t_tex = torch.tensor(tex_img, device=device).unsqueeze(0).contiguous()  # (1, H_tex, W_tex, 3)
tex_original = t_tex.clone()  # Save for comparison
t_tex = torch.nn.Parameter(t_tex)

# Reference images
t_refs = {n: torch.tensor(img, device=device).unsqueeze(0) for n, img in ref_imgs.items()}

# MVP matrices
proj = make_projection(sfx, sfy, scx, scy, RW, RH).to(device)
mvps = {}
for name, view in view_mats.items():
    mvps[name] = proj @ torch.tensor(view, device=device)

glctx = dr.RasterizeCudaContext()

# Quick sanity check: render one view
test_name = view_names[0]
with torch.no_grad():
    test_rgb, test_mask = render(glctx, t_pos, t_faces, t_uv, t_tex, mvps[test_name], [RH, RW])
    coverage = test_mask.sum() / (RH * RW) * 100
print(f"Sanity: rendered {test_name}, coverage={coverage:.1f}%")
print(f"GPU: {torch.cuda.memory_allocated()/1e6:.0f} MB")

In [ ]:
# Cell 6: Optimize texture via differentiable rendering
NUM_ITERS = 300
BATCH = 4          # Views per iteration
LR = 0.01
TV_WEIGHT = 1e-3   # Total-variation regularizer

optimizer = torch.optim.Adam([t_tex], lr=LR)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, NUM_ITERS, eta_min=LR * 0.1)

losses = []
resolution = [RH, RW]
rng = np.random.default_rng(42)

print(f"Optimizing: {NUM_ITERS} iters × {BATCH} views @ {RW}x{RH}")
print(f"Texture: {t_tex.shape[1]}x{t_tex.shape[2]}, {len(view_names)} views available")

for it in range(NUM_ITERS):
    optimizer.zero_grad()

    batch = rng.choice(view_names, size=min(BATCH, len(view_names)), replace=False)
    total_loss = torch.tensor(0.0, device=device)

    for name in batch:
        rendered, mask = render(glctx, t_pos, t_faces, t_uv, t_tex, mvps[name], resolution)
        ref = t_refs[name]
        # L1 on visible pixels
        diff = (rendered - ref) * mask
        total_loss = total_loss + diff.abs().sum() / mask.sum().clamp(min=1)

    total_loss = total_loss / len(batch)
    reg = tv_loss(t_tex) * TV_WEIGHT
    (total_loss + reg).backward()
    optimizer.step()
    scheduler.step()

    with torch.no_grad():
        t_tex.data.clamp_(0, 1)

    losses.append(total_loss.item())
    if (it + 1) % 50 == 0:
        print(f"  [{it+1:3d}/{NUM_ITERS}] L1={total_loss.item():.4f}  tv={reg.item():.5f}  lr={scheduler.get_last_lr()[0]:.5f}")

print(f"\nDone. L1: {losses[0]:.4f} → {losses[-1]:.4f}  ({(1-losses[-1]/losses[0])*100:.1f}% reduction)")

In [ ]:
# Cell 7: Visualize results
import matplotlib.pyplot as plt

# Pick 3 evenly-spaced views for comparison
n = len(view_names)
sample_names = [view_names[i] for i in [0, n // 3, 2 * n // 3]]

fig, axes = plt.subplots(4, 3, figsize=(18, 22))

# Row 0: Loss curve + texture atlas comparison
axes[0, 0].plot(losses, linewidth=1)
axes[0, 0].set_title('L1 Loss')
axes[0, 0].set_xlabel('Iteration')
axes[0, 0].grid(True, alpha=0.3)

# Crop a detail region of the texture to show the difference
th, tw = tex_original.shape[1], tex_original.shape[2]
cy_crop, cx_crop, sz = th // 3, tw // 3, min(th, tw) // 3
orig_crop = tex_original[0, cy_crop:cy_crop+sz, cx_crop:cx_crop+sz].cpu().numpy()
refi_crop = t_tex.data[0, cy_crop:cy_crop+sz, cx_crop:cx_crop+sz].cpu().numpy()
axes[0, 1].imshow(orig_crop)
axes[0, 1].set_title('Texture crop (original)')
axes[0, 2].imshow(refi_crop)
axes[0, 2].set_title('Texture crop (refined)')

# Rows 1-3: Per-view comparison (reference | original render | refined render)
for row, name in enumerate(sample_names, start=1):
    with torch.no_grad():
        rend_orig, mask = render(glctx, t_pos, t_faces, t_uv, tex_original, mvps[name], resolution)
        rend_new, _ = render(glctx, t_pos, t_faces, t_uv, t_tex, mvps[name], resolution)

    axes[row, 0].imshow(t_refs[name][0].cpu().numpy())
    axes[row, 0].set_title(f'Reference: {name[:20]}...')
    axes[row, 1].imshow(rend_orig[0].cpu().numpy())
    axes[row, 1].set_title('Rendered (original)')
    axes[row, 2].imshow(rend_new[0].cpu().numpy())
    axes[row, 2].set_title('Rendered (refined)')

for ax in axes.flat:
    ax.axis('off')
axes[0, 0].axis('on')

plt.tight_layout()
plt.savefig('/content/comparison.png', dpi=150)
plt.show()
print("Saved /content/comparison.png")

In [ ]:
# Cell 8: Save refined texture + download
import shutil, zipfile

OUT = '/content/nvdiffrast_output'
os.makedirs(OUT, exist_ok=True)

# Save refined texture (same filename as original → drop-in replacement)
tex_basename = os.path.basename(tex_file)
refined_np = (t_tex.data[0].cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
Image.fromarray(refined_np).save(os.path.join(OUT, tex_basename))

# Copy OBJ + MTL (geometry/UVs unchanged)
shutil.copy(obj_path, OUT)
shutil.copy(mtl_path, OUT)
shutil.copy('/content/comparison.png', OUT)

# Also save original for A/B comparison
orig_np = (tex_original[0].cpu().numpy() * 255).clip(0, 255).astype(np.uint8)
Image.fromarray(orig_np).save(os.path.join(OUT, 'original_' + tex_basename))

# Zip
zip_path = '/content/nvdiffrast_output.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for root, dirs, fnames in os.walk(OUT):
        for fn in fnames:
            fp = os.path.join(root, fn)
            z.write(fp, os.path.relpath(fp, OUT))

print(f"Output: {zip_path}")
for f in sorted(os.listdir(OUT)):
    sz = os.path.getsize(os.path.join(OUT, f)) / 1e6
    print(f"  {f} ({sz:.1f} MB)")

files.download(zip_path)